
# 04 - Hyperparameter Search

## Decisiones tomadas




In [ ]:
print("done")

In [ ]:

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from PIL import Image

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

from torch.utils.tensorboard import SummaryWriter

print("Imports OK")


In [ ]:

class SkinDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = root_dir
        self.transform = transform

        self.image_paths = []
        self.labels = []

        self.class_names = sorted(os.listdir(root_dir))

        self.class_to_idx = {
            cls_name: idx
            for idx, cls_name in enumerate(self.class_names)
        }

        for class_name in self.class_names:

            class_dir = os.path.join(root_dir, class_name)

            if not os.path.isdir(class_dir):
                continue

            for fname in os.listdir(class_dir):

                if fname.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".bmp", ".webp")
                ):

                    self.image_paths.append(
                        os.path.join(class_dir, fname)
                    )

                    self.labels.append(
                        self.class_to_idx[class_name]
                    )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]

        image = Image.open(image_path).convert("RGB")

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


print("Dataset class OK")


In [ ]:

train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2
    ),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
])

print("Transforms OK")


In [ ]:

train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val"

train_dataset = SkinDataset(
    train_dir,
    train_transform
)

val_dataset = SkinDataset(
    val_dir,
    val_transform
)

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))

print("Classes:")
print(train_dataset.class_names)


In [ ]:
class BaselineCNN(nn.Module):

    def __init__(self, num_classes, dropout=0.5):

        super(BaselineCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Opcional: Dropout sobre mapas de características
            nn.Dropout2d(0.25)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 16 * 16,
                256
            ),

            nn.ReLU(),

            # Dropout clásico
            nn.Dropout(dropout),

            nn.Linear(
                256,
                num_classes
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

num_classes = len(train_dataset.class_names)

model = BaselineCNN(num_classes)

model = model.to(device)

print(model)


In [ ]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

writer = SummaryWriter("runs/hyperparameter_cnn")

n_epochs = 10

best_accuracy = 0.0

for epoch in range(n_epochs):

    model.train()

    running_loss = 0.0

    progress_bar = tqdm(train_loader)

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        progress_bar.set_description(
            f"Epoch {epoch+1}/{n_epochs} - Loss: {loss.item():.4f}"
        )

    epoch_loss = running_loss / len(train_loader)

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    writer.add_scalar(
        "Loss/train",
        epoch_loss,
        epoch
    )

    writer.add_scalar(
        "Accuracy/validation",
        accuracy,
        epoch
    )

    print(
        f"Epoch {epoch+1}: "
        f"Train Loss={epoch_loss:.4f} | "
        f"Validation Accuracy={accuracy:.4f}"
    )

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        torch.save(
            model.state_dict(),
            "hyperparameter_cnn_best.pth"
        )

        print("Nuevo mejor modelo guardado")

print("\nEntrenamiento finalizado")
print(f"Best Validation Accuracy: {best_accuracy:.4f}")


In [ ]:

model.load_state_dict(
    torch.load("hyperparameter_cnn_best.pth")
)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_preds.extend(
            predicted.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=train_dataset.class_names
    )
)

cm = confusion_matrix(
    all_labels,
    all_preds
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=train_dataset.class_names,
    yticklabels=train_dataset.class_names
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.show()



## TensorBoard

Para visualizar métricas de entrenamiento:

```bash
tensorboard --logdir=runs
```

Luego abrir:

```text
http://localhost:6006
```


In [ ]:

# Opcional:
# Ejecutar TensorBoard dentro del notebook

# %load_ext tensorboard
# %tensorboard --logdir runs



# Conclusiones



## Hyperparameter Search

In [ ]:

learning_rates=[1e-2,1e-3,1e-4]
batch_sizes=[16,32,64]
dropouts=[0.3,0.5]
optimizers=["Adam","SGD"]

results=[]
best_accuracy=0
best_config=None

for lr in learning_rates:
    for batch_size in batch_sizes:
        for dropout in dropouts:
            for opt_name in optimizers:
                print(f"Training lr={lr}, batch={batch_size}, dropout={dropout}, opt={opt_name}")
                train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
                val_loader=DataLoader(val_dataset,batch_size=batch_size,shuffle=False)
                model=BaselineCNN(num_classes,dropout=dropout).to(device)
                criterion=nn.CrossEntropyLoss()
                if opt_name=="Adam":
                    optimizer=optim.Adam(model.parameters(),lr=lr)
                else:
                    optimizer=optim.SGD(model.parameters(),lr=lr,momentum=0.9)
                best_val=0
                for epoch in range(10): # ajustar épocas
                    model.train()
                    for images,labels in train_loader:
                        images,labels=images.to(device),labels.to(device)
                        optimizer.zero_grad()
                        loss=criterion(model(images),labels)
                        loss.backward()
                        optimizer.step()
                    model.eval()
                    correct=total=0
                    with torch.no_grad():
                        for images,labels in val_loader:
                            images,labels=images.to(device),labels.to(device)
                            pred=model(images).argmax(1)
                            correct+=(pred==labels).sum().item()
                            total+=labels.size(0)
                    acc=100*correct/total
                    best_val=max(best_val,acc)
                results.append({"lr":lr,"batch_size":batch_size,"dropout":dropout,"optimizer":opt_name,"accuracy":best_val})
                if best_val>best_accuracy:
                    best_accuracy=best_val
                    best_config=results[-1]
                    torch.save(model.state_dict(),"best_model.pth")
import pandas as pd
df=pd.DataFrame(results).sort_values("accuracy",ascending=False)
print(df)
print("Best:",best_config)
